# Part 1 : Setting Up Zingg
## It is responsible for initializing the Zingg environment, which includes the following steps:
- **Environment Setup:** Loads all necessary libraries and dependencies required for Zingg to run.
- **Path Setup:** Defines and sets up all relevant file paths, such as model directory, input data locations, output directories.
- **Performance Tuning:** Applies Spark and Zingg performance-related configurations to optimize the execution of data processing tasks.

## Example Notebook For Training and Running Zingg Entity Resolution Workflow on Fabric
This notebook runs the Zingg Febrl Example on Fabric. Please refer to the

- Zingg Python API
- Zingg Official Documentation for details.

_This notebook has been tested on Runtime 1.3 version (Spark 3.5)_

## Create a environment and install Zingg
# 
- Go to the Clusters tab, hit Create Cluster, and give it a name like “Zingg.”
- Set the runtime version to a current LTS (Long-Term Support) version for compatibility.
- Next, you’ll need to install Zingg. For this, we will be need the latest Zingg JAR file.
- You need to go to the Environment tab and click on the New Environment button. You can name the new Environment "Zingg Environment".
- Visit [Zingg releases](https://github.com/zinggAI/zingg/releases), find the latest version of Zingg, and download the tar file and Extract the jar file from the newly downloaded tar file.
- you need to open the Environment, that you have created earlier. Then go to the custom library and upload the jar file there.
-Now you need to go back to the Environment “Zingg Environment”, click the “new item” button and select Lakehouse inside of it.
- Zingg supports multiple file formats like CSV, Parquet, or JSON. For this example, let’s use a CSV file. You now need to go inside the Lakehouse, click on “Get data,” and upload the CSV file.
- Save and Publish the files

### Please execute each cell one by one as per the instructions provided.


## Install Zingg
## Check if all the Zingg wheels are installed correctly

In [ ]:
pip install zingg

In [ ]:
!pip show zingg

## Set Checkpoint directory

In [ ]:
spark.sparkContext.setCheckpointDir("Files/checkpoint")

## Import the required libraries

In [ ]:
## Import necessary libraries
import pandas as pd
import numpy as np
from ipywidgets import widgets, interact, GridspecLayout
import base64
import pyspark.sql.functions as fn


# Zingg libraries
from zingg.client import *
from zingg.pipes import *

# Function to count labeled pairs
def count_labeled_pairs(marked_pd):
    '''
    The purpose of this function is to count the labeled pairs in the marked folder.
    '''
    n_total = len(np.unique(marked_pd['z_cluster']))
    n_positive = len(np.unique(marked_pd[marked_pd['z_isMatch'] == 1]['z_cluster']))
    n_negative = len(np.unique(marked_pd[marked_pd['z_isMatch'] == 0]['z_cluster']))
    n_uncertain = len(np.unique(marked_pd[marked_pd['z_isMatch'] == 2]['z_cluster']))

    return n_positive, n_negative, n_uncertain, n_total

# Setup interactive widget
available_labels = {
    'No Match': 0,
    'Match': 1,
    'Uncertain': 2
}


## Define locations for the model
The Zingg models and training data are persisted in storage.

Please edit the model id in the cell below to reflect your model.

In [ ]:
##you can change these to the locations of your choice
##these are the only two settings that need to change
zinggDir = "abfss://ACD@onelake.dfs.fabric.microsoft.com/newlk.Lakehouse/Files/models"
modelId = "oss11Dec"


## Set the Directories path

In [ ]:
## Define constants
MARKED_DIR = zinggDir + "/" + modelId + "/trainingData/marked/"
UNMARKED_DIR = zinggDir + "/" + modelId + "/trainingData/unmarked/"


## Start building the Zingg program
The following cell sets up the initial arguments for Zingg.

In [ ]:
#build the arguments for zingg
args = Arguments()
# Set the modelid and the zingg dir. You can use this as is
args.setModelId(modelId)
args.setZinggDir(zinggDir)

## Performance settings
The numPartitions define how data is split across the cluster. Please change this as per your data and cluster size 

For details, refer to [Zingg performance tuning documentation](https://docs.zingg.ai/latest/stepbystep/configuration/tuning-label-match-and-link-jobs).
In general:
- keep `numPartitions` to ~20-30x the worker vCPU count 
- Disable Spark's Adaptive Query Execution

__NOTE__: *Please modify this for your use case*

In [ ]:
args.setNumPartitions(4)
args.setLabelDataSampleSize(0.4)


In [ ]:
spark.conf.set("spark.sql.adaptive.enabled", False)

## Define the input
Please refer to [Pipes](https://docs.zingg.ai/latest/connectors/pipes) for details on different formats.

Please modify this for your data.

In [ ]:
# Import pandas
import pandas as pd

# Define the schema (optional for validation)
schema = ["id", "fname", "lname", "stNo", "add1", "add2", "city", "areacode", "state", "dob", "ssn"]

# Load the CSV file
data = pd.read_csv("abfss://ACD@onelake.dfs.fabric.microsoft.com/newlk.Lakehouse/Files/test.csv",header=None)

# Ensure column names match the schema
data.columns = schema  # Adjust only if the file's column names differ

# Display the data
data.head()

In [ ]:
schema = "rec_id string, fname string, lname string, stNo string, add1 string, add2 string, city string, areacode string, state string, dob string, ssn string"
inputPipe = CsvPipe("inputpipe", "abfss://ACD@onelake.dfs.fabric.microsoft.com/newlk.Lakehouse/Files/test.csv", schema)

args.setData(inputPipe)

# Configure the output
Here we configure the output to be a csv, but similar to the input above, the output can be a file format like parquet or delta or a data store like MySQL

**Please modify this for your data.**

In [ ]:
#setting outputpipe in 'args'
output_path = "abfss://ACD@onelake.dfs.fabric.microsoft.com/newlk.Lakehouse/Files/ossOutput"+modelId
outputPipe = CsvPipe("resultOutput", output_path)
args.setOutput(outputPipe)

# Define the match fields and their types

The cell below is used to configure Zingg with the fields for use in matching and the match types.
Details on the field definitions can be found at [Zingg official docs](https://docs.zingg.ai/latest)

**Please modify this for your data.**

In [ ]:
# Set field definitions
rec_id = FieldDefinition("rec_id", "string", MatchType.DONT_USE)        
fname = FieldDefinition("fname", "string", MatchType.FUZZY)  # First Name
lname = FieldDefinition("lname", "string", MatchType.FUZZY)  # Last Name
stNo = FieldDefinition("stNo", "string", MatchType.FUZZY)    # Street Number
add1 = FieldDefinition("add1", "string", MatchType.FUZZY)    # Address Line 1
add2 = FieldDefinition("add2", "string", MatchType.FUZZY)    # Address Line 2
city = FieldDefinition("city", "string", MatchType.FUZZY)    # City
areacode = FieldDefinition("areacode", "string", MatchType.FUZZY)    # areacode
state = FieldDefinition("state", "string", MatchType.FUZZY)  # State
dob = FieldDefinition("dob", "string", MatchType.EXACT)      # Date of Birth (prefer exact match)
ssn = FieldDefinition("ssn", "string", MatchType.EXACT)      # SSN (should use exact match)

# Create the field definitions list
fieldDefs = [rec_id, fname, lname, stNo, add1, add2, city, areacode, state, dob, ssn]

# Set field definitions in args
args.setFieldDefinition(fieldDefs)